# COIN on TSP and Multi-objective TSP (up to 24 cities)
This library example uses deterministic teaching fixtures. A chromosome is a permutation of city IDs; evaluation closes the tour from the last city back to the first. All objectives are minimized.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from coin.core import PermutationCoinAlgorithm, MultiObjectiveCoinAlgorithm
from coin.models import (
    EdgeConfig, OptimizedEdgeCoin, RoseConfig,
    ROSE, ROSESingleRef, ROSESingleRefHistogram,
    TemplateROSE, TemplateROSESingleRef, TemplateROSESingleRefHistogram,
)
from coin.problems.tsp import get_tsp_instance, list_tsp_instances, TSPProblem
[(x.id, x.dimension, x.available_objective_names) for x in list_tsp_instances()]

## Single-objective TSP
Run Edge COIN on the preloaded 16-city distance instance and inspect best-so-far learning progress.

In [ ]:
instance = get_tsp_instance('tsp-16')
problem = TSPProblem(instance, ('distance',))
config = EdgeConfig(problem_size=problem.dimension, population_size=100, reward_ratio=10, punishment_ratio=10, training_rate=5, objective='min')
algorithm = PermutationCoinAlgorithm(OptimizedEdgeCoin(config, seed=1), problem)
algorithm.run(100)
best_so_far = np.minimum.accumulate([point.best for point in algorithm.history])
plt.plot([point.evaluations for point in algorithm.history], best_so_far)
plt.xlabel('objective evaluations'); plt.ylabel('best distance (lower is better)'); plt.grid(alpha=.25);

## ROSE: Multi-Reference Mean versus Single-Reference Range
ROSE learns exact node positions and compact signed-distance statistics for every ordered city pair. `ROSE-MultiRef Mean` preserves the original baseline by averaging predictions from several references. `ROSE-SingleRef Range` selects one actual reference and samples a truncated distance using its mean, minimum, maximum, and standard deviation. Template variants keep part of a parent tour and regenerate only punched positions. The following comparison gives every variant the same problem, population, generations, and seed.

In [ ]:
rose_variants = {
    'ROSE-MultiRef Mean': (ROSE, 'mean', False),
    'ROSE-SingleRef Range': (ROSESingleRef, 'uniform', False),
    'Template-ROSE-MultiRef Mean': (TemplateROSE, 'mean', True),
    'Template-ROSE-SingleRef Range': (TemplateROSESingleRef, 'uniform', True),
    'ROSE-SingleRef Histogram': (ROSESingleRefHistogram, 'uniform', False),
    'Template-ROSE-SingleRef Histogram': (TemplateROSESingleRefHistogram, 'uniform', True),
}
rose_runs = {}
for name, (model_class, reference_selection, template_enabled) in rose_variants.items():
    rose_config = RoseConfig(
        problem_size=problem.dimension, population_size=40, selection_ratio=20,
        reference_selection=reference_selection, node_weight=0.25,
        roll_mode='random', max_roll=5, smoothing=1.0, temperature=1.0,
        template_enabled=template_enabled, template_sample_ratio=50, objective='min',
    )
    rose_algorithm = PermutationCoinAlgorithm(model_class(rose_config, seed=1), problem)
    rose_algorithm.run(30)
    rose_runs[name] = (rose_algorithm, rose_algorithm.model.diagnostics())

for name, (run, diagnostics) in rose_runs.items():
    best = min(point.best for point in run.history)
    print(f'{name:28s} best={best:8.2f}  evaluations={run.evaluations:4d}  '
          f'duplicate_rate={diagnostics["duplicate_rate"]:.3f}')

In [ ]:
for name, (run, _) in rose_runs.items():
    values = np.minimum.accumulate([point.best for point in run.history])
    plt.step([point.evaluations for point in run.history], values, where='post', label=name)
plt.xlabel('objective evaluations')
plt.ylabel('best distance (lower is better)')
plt.title('Fair ROSE comparison on the same TSP instance')
plt.grid(alpha=.25); plt.legend();

### Inspecting what ROSE learned
`pair_statistics(i, j)` reports mean, minimum, maximum, SD, and sample count between two cities. Diagnostics expose estimator memory, reference-selection frequency, sampled versus realized distance, fallbacks, offspring diversity, and template usage. All six estimator matrices are `n × n`, so memory grows as `O(n^2)`.

In [ ]:
single_ref_model = rose_runs['ROSE-SingleRef Range'][0].model
print('city 0 relative to city 1:', single_ref_model.pair_statistics(0, 1))
single_ref_model.diagnostics()

## Bi-objective MO-TSP
The two matrices represent geographic distance and operating cost. MO COIN learns by nondominated depth/diversity and retains a unique external Pareto archive.

In [ ]:
mo_instance = get_tsp_instance('motsp-16')
mo_problem = TSPProblem(mo_instance, ('distance', 'operating_cost'))
mo_config = EdgeConfig(problem_size=mo_problem.dimension, population_size=100, reward_ratio=10, punishment_ratio=10, training_rate=5, objective='min')
mo_algorithm = MultiObjectiveCoinAlgorithm(OptimizedEdgeCoin(mo_config, seed=1), mo_problem)
for _ in range(100): mo_algorithm.step()
front = mo_algorithm.archive_values
order = np.argsort(front[:, 0])
plt.scatter(front[:, 0], front[:, 1], label='nondominated tours')
plt.plot(front[order, 0], front[order, 1], alpha=.5)
plt.xlabel('distance'); plt.ylabel('operating cost'); plt.grid(alpha=.25); plt.legend();
print('archive size:', len(front), 'evaluations:', mo_algorithm.evaluations)

## Reproducible comparison checklist
Use the same `TSPProblem` object, population, generations/evaluation budget, and seeds for every algorithm. Report every seed, best/mean for scalar TSP, and convergence, spread, hypervolume or nondominated ratio for MO-TSP. The preloads are controlled teaching fixtures—not claims of published best-known benchmarks.